# Temporal difference learning

En el Aprendizaje por Refuerzo, TD (Temporal Difference) Learning es considerado por muchos como la idea central y más novedosa de toda la disciplina. Es un híbrido que combina lo mejor de dos mundos: la **Programación Dinámica (DP)** y los **métodos de Monte Carlo (MC)**.

## ¿Qué es el Temporal-Difference Learning?

*TD Learning* es un método para que el agente aprenda a predecir la suma de recompensas futuras (la función de valor) basándose en su experiencia, pero **sin tener que esperar a que termine el episodio**. Es una evolución de los métodos de Monte Carlo, que necesitaban terminar el episodio para saber la recompensa y actualizar la función valor.

Una analogía clásica para explicar la diferencia entre MC y TD es predecir el tiempo que vas a tardar en llegar a tu destino en coche:
- *Enfoque Monte Carlo*: Conduces todo el camino, llegas a tu destino y dices: *"Vale, he tardado 45 minutos. Mi estimación inicial de 30 minutos era mala. La actualizaré para la próxima vez"*.
- *Enfoque TD*: Sales de casa estimando tardar 30 minutos. A los 5 minutos, te encuentras con un atasco terrible. En ese mismo instante, sin haber llegado a tu destino, actualizas tu predicción: *"Vaya, con este atasco voy a tardar al menos 50 minutos"*. Has actualizado tu predicción usando tu nueva realidad + tu estimación del resto del camino.

## Formulación matemática

En Monte Carlo, para actualizar el valor de un estado $V_\pi(s)$, esperábamos a conocer el retorno total real al final del episodio ($G$).

En TD, reemplazamos ese retorno final $G$ por una estimación calculada tras dar un solo paso. La ecuación que estima la recompensa esperada mediante TD es:

$$
V_\pi(s) \doteq \mathbb{E} \left [ V_\pi(s) + \alpha (r + \gamma V_\pi(s') + V_\pi(s)) | s \right ]
$$

Suponiendo que la media exponencialmente ponderada es un estimador adecuado para la esperanza $\mathbb{E}$, podemos convertir dicha ecuación en una regla de actualización para un algoritmo online:

$$
V_\pi(s) \leftarrow V_\pi(s) + \alpha \left ( r + \gamma V_\pi(s') - V_\pi(s) \right )
$$

Desglosemos sus componentes clave:
- $V_\pi(s)$: La estimación actual del estado en el que estamos
- $\alpha$: El tamaño de paso (*learning rate*).
- $r + \gamma V_\pi(s')$: A esto se le llama el **Objetivo TD (TD Target)**. Es la recompensa real que acabamos de recibir, más la estimación descontada del estado en el que hemos aterrizado.
- $r + \gamma V_\pi(s') - V_\pi(s)$: A esto se le llama el **Error TD (TD Error)**. Es la diferencia entre lo que pensábamos que iba a pasar y lo que acaba de pasar empíricamente en este paso.

## Diferencia Temporal: TD(0) en la Pasarela

Vamos a implementar esto en nuestra pasarela de 5 casillas. Veremos cómo el agente va corrigiendo sus estimaciones en directo, sin necesidad de terminar el recorrido.

In [ ]:
import numpy as np

def estimar_V_TD0(num_episodios: int, alpha: float = 0.1, gamma: float = 1.0) -> dict[int, float]:
    # Inicializamos la función de valor V(s) a 0.0 para los estados no terminales (0, 1, 2, 3)
    # El estado terminal (4) ya sabemos que vale 5, y salir por la izquierda vale 0.
    V: dict[int, float] = {s: 0.0 for s in range(4)}
    
    for i in range(num_episodios):
        estado = 0  # Empezamos siempre en la casilla 0
        terminado = False
        
        while not terminado:
            # Nuestra política: moverse aleatoriamente
            accion = np.random.choice([-1, 1])
            estado_siguiente = estado + accion
            
            # Recompensa y valor del siguiente estado
            if estado_siguiente < 0:
                recompensa = 0
                v_siguiente = 0.0
                terminado = True
            elif estado_siguiente > 3:
                recompensa = 5
                v_siguiente = 0.0 # Es terminal, su valor intrínseco posterior es 0
                terminado = True
            else:
                recompensa = 0
                v_siguiente: float = V[estado_siguiente]
                terminado = False
                
            # Regla de actualización TD(0)
            objetivo_td: float = recompensa + gamma * v_siguiente
            error_td: float = objetivo_td - V[estado]
            
            V[estado] = V[estado] + alpha * error_td
            
            if not terminado:
                estado = estado_siguiente
            
        # Imprimimos el progreso para ver cómo converge paso a paso
        if (i + 1) % 200 == 0:
            valores_actuales: list[float] = [round(V[s], 2) for s in range(4)]
            print(f"Episodio {i + 1:^4}: {valores_actuales}")
            
    return V

# Entrenamos usando TD(0)
print("Convergencia de V(s) usando TD(0):")
valores_TD = estimar_V_TD0(num_episodios=2000, alpha=0.1)

Convergencia de V(s) usando TD(0):
Episodio 200 : [0.4, 2.19, 2.89, 4.03]
Episodio 400 : [1.3, 2.3, 3.41, 4.56]
Episodio 600 : [1.34, 2.62, 3.75, 4.58]
Episodio 800 : [0.95, 1.79, 2.73, 3.61]
Episodio 1000: [0.82, 2.03, 3.12, 4.16]
Episodio 1200: [0.82, 2.26, 2.88, 4.25]
Episodio 1400: [0.7, 1.78, 3.21, 4.03]
Episodio 1600: [1.25, 1.93, 3.44, 4.16]
Episodio 1800: [0.82, 2.87, 3.88, 4.47]
Episodio 2000: [0.66, 1.54, 2.69, 3.75]


## Ventajas de TD
Las principales ventajas de Temporal Difference Learning son las siguientes:
1. **Aprendizaje Online y Continuo**: A diferencia de la pasarela, ¿qué pasa si entrenamos un robot para caminar o un sistema de trading en bolsa que nunca termina? Monte Carlo se atascaría porque nunca llegaría al final del episodio para aprender. TD aprende en cada tick del reloj.
1. **Menor Varianza**: Monte Carlo sufre de mucha varianza porque un episodio largo tiene muchas decisiones y azar acumulado. TD reduce esa varianza drásticamente al basarse solo en la aleatoriedad de un único paso.

# Q-Learning: Aprendizaje de Diferencia Temporal para Acciones

Hasta ahora hemos evaluado lo bueno que es un estado, $V(S)$. Pero para tomar decisiones sin conocer las reglas del entorno, necesitamos estimar el valor de las acciones, $Q(S, A)$.

**Q-Learning** es un algoritmo de control basado en Diferencia Temporal (TD). Su característica más importante es que es **Off-Policy (Fuera de Política)**. Esto significa que el agente puede estar explorando el entorno de forma aleatoria o subóptima (por ejemplo, usando $\epsilon$-greedy), pero *aprende* la tabla $Q$ asumiendo que en el futuro tomará la decisión absolutamente perfecta.

Partiendo de la regla de actualización de la función estado-valor de TD podemos obtener la regla de actualización de la función acción-valor Q-Learning es:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma \argmax_{a_s \in \mathfrak{A}(s)} Q(s', a_s) - Q(s, a) \right)$$

Desglosemos el **Objetivo TD** en esta fórmula: $r + \gamma \argmax_{a_s \in \mathfrak{A}(s)} Q(s', a_s)$
1. $r$: La recompensa que acabamos de recibir.
2. $\argmax_{a_s \in \mathfrak{A}(s)} Q(s', a_s)$: Miramos al nuevo estado $s'$ en el que hemos aterrizado y nos preguntamos: de todas las acciones posibles aquí $a_s \in \mathfrak{A}(s)$, ¿cuál es la que tiene el valor $Q$ más alto?". Tomamos ese valor máximo, lo multiplicamos por el descuento $\gamma$, y se lo sumamos a la recompensa.

Vamos a implementar Q-Learning en nuestra pasarela de 5 casillas.

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict

def elegir_accion_eps_greedy(estado: int, Q: dict, epsilon: float) -> str:
    """Política Epsilon-Greedy para seleccionar acciones."""
    if np.random.random() < epsilon:
        return np.random.choice(['Izquierda', 'Derecha'])

    q_izq = Q[estado]['Izquierda']
    q_der = Q[estado]['Derecha']

    if q_der > q_izq:
        return 'Derecha'
    elif q_izq > q_der:
        return 'Izquierda'
    else:
        return np.random.choice(['Izquierda', 'Derecha'])

def q_learning_pasarela(num_episodios: int, alpha: float = 0.1, gamma: float = 1.0, epsilon: float = 0.2, living_penalty: float = 0) -> Dict[int, Dict[str, float]]:
    # Inicializamos la Tabla Q a 0.0 para los estados 0, 1, 2, 3
    Q: dict[int, dict[str, float]] = {s: {'Izquierda': 0.0, 'Derecha': 0.0} for s in range(4)}
    movimientos: dict[str, int] = {'Izquierda': -1, 'Derecha': 1}

    for _ in range(num_episodios):
        estado = 0  # Inicio de la pasarela
        terminado = False
        while not terminado:
            # Elegimos la acción usando nuestra política (epsilon-greedy)
            accion: str = elegir_accion_eps_greedy(estado, Q, epsilon)
            estado_siguiente: int = estado + movimientos[accion]

            # Observamos la recompensa y el estado siguiente
            if estado_siguiente < 0:
                recompensa = 0
                max_q_siguiente = 0.0  # Estado terminal
                terminado = True
            elif estado_siguiente > 3:
                recompensa = 5
                max_q_siguiente = 0.0  # Estado terminal
                terminado = True
            else:
                recompensa = living_penalty
                # Tomamos el máximo valor Q del siguiente estado
                max_q_siguiente: float = max(Q[estado_siguiente].values())
                terminado = False

            # 3. Aplicamos la Ecuación de Actualización de Q-Learning
            objetivo_td: float = recompensa + gamma * max_q_siguiente
            error_td: float = objetivo_td - Q[estado][accion]

            Q[estado][accion] += alpha * error_td

            if not terminado:
                estado: int = estado_siguiente
    return Q

# Entrenamos a nuestro agente con Q-Learning
Q_final: dict[int, dict[str, float]] = q_learning_pasarela(num_episodios=2000, alpha=0.1, epsilon=0.2)

# Formateamos para visualizar como Tabla
df_q = pd.DataFrame.from_dict(Q_final, orient='index')
df_q.index.name = 'Estado (s)'
df_q.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']
print("Tabla Q aprendida mediante Q-Learning:")
display(df_q)

Tabla Q aprendida mediante Q-Learning:


,"Q(s, Izquierda)","Q(s, Derecha)"
Estado (s),,
0,0.0,5.0
1,5.0,5.0
2,5.0,5.0
3,5.0,5.0


## El factor de descuento y la penalización por paso

Mirando la tabla Q anterior se podría decir que al agente le da igual ir hacia la izquierda o la derecha en cualquier estado menos en el inicial, que tiene claro que no debe ir a la izquierda. Este comportamiento es completamente normal y se debe a que no hemos ajustado un factor de descuento de las recompensas futuras $\gamma$ ni hemos establecido una penalización por paso.

Al no haber penalización por paso, cada movimiento de una casilla a otra da una recompensa de 0. Al agente no le cuesta nada caminar, por lo que no tiene prisa.

Por otro lado, un factor de descuento $\gamma=1.0$ significa que al agente le da exactamente igual recibir 5 puntos ahora mismo, que recibirlos dentro de 10.000 pasos.

Para que un agente de RL busque el camino óptimo (el más corto), necesitamos introducir un sentido de urgencia. Tenemos dos formas de hacerlo:

### Factor de descuento $\gamma < 1.0$
Si cambiamos el valor del descuento, por ejemplo a $\gamma=0.9$, el valor futuro se degrada un 10% con cada paso. El agente preferirá ganar 5 puntos en 1 paso ($5 \times 0.9 = 4.5$) que en 2 pasos ($5 \times 0.9 \times 0.9=4.05$).

Reentrenamos el agente con un factor de descuento y observamos la tabla Q resultante:

In [ ]:
tabla_q_descuento: dict[int, dict[str, float]] = q_learning_pasarela(num_episodios=5000, alpha=0.1, gamma=0.5, epsilon=0.2)
# Formateamos para visualizar como Tabla
df_q_descuento = pd.DataFrame.from_dict(tabla_q_descuento, orient='index')
df_q_descuento.index.name = 'Estado (s)'
df_q_descuento.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']
print("Tabla Q aprendida mediante Q-Learning:")
display(df_q_descuento)

Tabla Q aprendida mediante Q-Learning:


,"Q(s, Izquierda)","Q(s, Derecha)"
Estado (s),,
0,0.0000,0.625
1,0.3125,1.250
2,0.6250,2.500
3,1.2500,5.000


### Coste de paso (living penalty)

En muchos juegos, en lugar de descontar el futuro, se le da al agente una recompensa negativa pequeña (ej. -0.1) por cada paso que da sin llegar a la meta. Esto le enseña que "estar vivo cuesta dinero" y le obliga a salir del laberinto lo antes posible.

Veamos cómo afecta esto a la tabla Q:

In [ ]:
tabla_q_paso: dict[int, dict[str, float]] = q_learning_pasarela(num_episodios=5000, alpha=0.1, gamma=1.0, epsilon=0.2, living_penalty=-0.1)
# Formateamos para visualizar como Tabla
df_q_paso = pd.DataFrame.from_dict(tabla_q_paso, orient='index')
df_q_paso.index.name = 'Estado (s)'
df_q_paso.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']
print("Tabla Q aprendida mediante Q-Learning:")
display(df_q_paso)

Tabla Q aprendida mediante Q-Learning:


,"Q(s, Izquierda)","Q(s, Derecha)"
Estado (s),,
0,0.0,4.7
1,4.6,4.8
2,4.7,4.9
3,4.8,5.0


Podemos estudiar qué pasa si combinamos ambos enfoques:

In [ ]:
tabla_q_ambos: dict[int, dict[str, float]] = q_learning_pasarela(num_episodios=5000, alpha=0.1, gamma=0.9, epsilon=0.2, living_penalty=-0.1)
# Formateamos para visualizar como Tabla
df_q_ambos = pd.DataFrame.from_dict(tabla_q_ambos, orient='index')
df_q_ambos.index.name = 'Estado (s)'
df_q_ambos.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']
print("Tabla Q aprendida mediante Q-Learning:")
display(df_q_ambos)

Tabla Q aprendida mediante Q-Learning:


,"Q(s, Izquierda)","Q(s, Derecha)"
Estado (s),,
0,0.0000,3.374
1,2.9366,3.860
2,3.3740,4.400
3,3.8600,5.000


# SARSA: On-Policy TD Control

**SARSA** debe su nombre a la secuencia de eventos que utiliza para actualizar su tabla: **S**tate, **A**ction, **R**eward, next **S**tate, next **A**ction ($S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1}$).

A diferencia de Q-Learning, que utiliza el operador $\max_{a'}$ para actualizar sus valores asumiendo que en el futuro jugará perfecto (Off-Policy), SARSA es un algoritmo **On-Policy**. Esto significa que actualiza la tabla Q basándose en la acción que *realmente* va a tomar en el siguiente paso, dictada por su política actual (por ejemplo, Epsilon-Greedy).

La ecuación de actualización es casi idéntica a la de Q-Learning, pero sin la trampa del máximo:

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma Q(S', A') - Q(S, A) \right]$$

**¿Por qué usar SARSA en lugar de Q-Learning?**
Porque SARSA es más "conservador". Al tener en cuenta que de vez en cuando va a explorar aleatoriamente (debido a $\epsilon$), SARSA evitará caminos que sean teóricamente óptimos pero que tengan peligros muy grandes cerca (como caer por un precipicio si el agente hace un movimiento aleatorio por error).

Vamos a implementar SARSA en nuestra pasarela. Fijaos muy bien en cómo ahora necesitamos elegir la Acción Siguiente ($A'$) **antes** de poder actualizar la tabla.

In [ ]:
import numpy as np
import pandas as pd

def sarsa_pasarela(num_episodios: int, alpha: float = 0.1, gamma: float = 0.9, epsilon: float = 0.2):
    Q = {s: {'Izquierda': 0.0, 'Derecha': 0.0} for s in range(4)}
    movimientos = {'Izquierda': -1, 'Derecha': 1}
    
    for _ in range(num_episodios):
        estado = 0
        
        # SARSA necesita elegir la PRIMERA acción antes de entrar al bucle
        accion = elegir_accion_eps_greedy(estado, Q, epsilon)

        terminado = False

        while not terminado:
            estado_siguiente = estado + movimientos[accion]
            
            # Observamos la recompensa y el estado siguiente
            if estado_siguiente < 0:
                recompensa = 0
                q_siguiente = 0.0
                terminado = True
            elif estado_siguiente > 3:
                recompensa = 5
                q_siguiente = 0.0
                terminado = True
            else:
                recompensa = 0
                # Elegimos la SIGUIENTE acción usando la MISMA política (On-Policy)
                accion_siguiente = elegir_accion_eps_greedy(estado_siguiente, Q, epsilon)
                
                # Usamos el valor Q de la acción que realmente hemos elegido (no hacemos la 'trampa' de elegir la máxima recompensa posible)
                q_siguiente = Q[estado_siguiente][accion_siguiente]
                terminado = False
                
            # Aplicamos la Ecuación de Actualización de SARSA
            objetivo_td = recompensa + gamma * q_siguiente
            Q[estado][accion] += alpha * (objetivo_td - Q[estado][accion])
                
            if not terminado:
                # Avanzamos: El estado siguiente se convierte en el actual...
                # Y la acción siguiente que pre-calculamos se convierte en la acción a ejecutar
                estado = estado_siguiente
                accion = accion_siguiente 
            
    return Q

# Entrenamos a nuestro agente con SARSA
Q_final_sarsa = sarsa_pasarela(num_episodios=5000, alpha=0.1, gamma=0.9, epsilon=0.2)

df_q_sarsa = pd.DataFrame.from_dict(Q_final_sarsa, orient='index')
df_q_sarsa.index.name = 'Estado (s)'
df_q_sarsa.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']
print("Tabla Q aprendida mediante SARSA:")
display(df_q_sarsa)

Tabla Q aprendida mediante SARSA:


,"Q(s, Izquierda)","Q(s, Derecha)"
Estado (s),,
0,0.000000,3.412482
1,2.262307,3.813470
2,3.254226,4.331966
3,3.763840,5.000000


# Resumen y Comparativa: Q-Learning vs. SARSA

Aunque ambos son algoritmos de control basados en Diferencia Temporal (TD), la pequeña diferencia en su regla de actualización (usar el máximo teórico vs. usar la acción real) provoca comportamientos radicalmente distintos en los agentes.

| Característica | Q-Learning (Off-Policy) | SARSA (On-Policy) |
| :--- | :--- | :--- |
| **Definición** | Aprende la política óptima independientemente de las acciones exploratorias del agente. | Aprende el valor de la política que el agente está ejecutando actualmente (incluyendo sus errores). |
| **Actualización (TD Target)** | Utiliza $\max_{a'} Q(S', a')$ (El mejor escenario posible). | Utiliza $Q(S', A')$ (La acción que realmente tomará). |
| **Comportamiento** | **Arriesgado / Codicioso**. Busca el camino más rápido sin importarle si pasa cerca del peligro. | **Conservador / Seguro**. Mantiene distancia con los peligros porque es "consciente" de que puede equivocarse explorando. |
| **Principal Ventaja** | Si se le da el tiempo suficiente, garantiza encontrar la ruta matemáticamente perfecta. | Mejor rendimiento y menos penalizaciones *durante* la fase de entrenamiento (ideal para robots físicos reales). |
| **Principal Inconveniente** | En entornos con penalizaciones severas, sufrirá muchísimos accidentes mientras entrena debido a la exploración aleatoria. | Puede aprender una ruta más larga y subóptima (por miedo al peligro) si el ratio de exploración $\epsilon$ no decae con el tiempo. |

**¿Cuándo usar cada uno?**
* Usa **Q-Learning** si entrenas en un simulador donde "morir" es gratis y lo único que te importa es extraer la política óptima final.
* Usa **SARSA** si estás entrenando un agente en un entorno real u online donde los errores durante el entrenamiento cuestan dinero o pueden dañar el equipo.